# RI-JK UHF Hessian：交换分解

本文档与 `02-3-decomp_de_K.ipynb` 几乎完全相同，区别在于使用 UHF 的密度矩阵分解。

**UHF 与 RHF 的关键区别**：

- UHF 交换能 $E_K = -\frac{1}{2}\sum_\sigma (\mu\nu|\kappa\lambda) D_{\mu\kappa}^\sigma D_{\nu\lambda}^\sigma$，**按自旋分别构造**（与 J 只依赖总密度不同，K 不能简单地用总密度代入）。
- 因此 K 的每个 skeleton 子项需要 **对 $\alpha, \beta$ 两个自旋通道分别计算，再求和**。
- 形式上把 RHF 的 `mocc_2`（带 $\sqrt{\text{occ}=2}$ 因子的占据轨道系数）替换为 UHF 中各自旋通道的 `mocc[σ]`（UHF 中 occ=1，因此无需 $\sqrt{\cdot}$ 因子）；其他 einsum 结构与系数完全保持不变。
- 与 04 中存储的 `de_K20/de_K11/de_K02` 一致：PySCF 的 UHF `ek_aux*` 已经包含两个自旋通道之和，无需额外 ×2，所以本 notebook 中每个子项的系数与 RHF 完全相同（譬如 K20_1a/1b/2/3 均为 2；K11 的 2/2/−2/2；K02 的 1/−1/−0.5/−0.5/−1/0.5/0.5/−1/1），仅是 einsum 末端的占据轨道改为按自旋分量分别求和。

In [1]:
from pyscf import gto, scf, lib
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", charge=2, spin=2, max_memory=32000).build()

In [3]:
mf = scf.UHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_u_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_u_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_u_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_K20 = np.load("nh3_u_hf_decomp.npz")["de_K20"]
de_K11 = np.load("nh3_u_hf_decomp.npz")["de_K11"]
de_K02 = np.load("nh3_u_hf_decomp.npz")["de_K02"]

In [6]:
# UHF 索引约定：spin 维度记号 `x`，0 = alpha, 1 = beta
α, β = 0, 1

mo_coeff = mf.mo_coeff           # shape (2, nao, nmo)
mo_occ = mf.mo_occ               # shape (2, nmo)
mo_energy = mf.mo_energy         # shape (2, nmo)
nao = mo_coeff.shape[1]
nmo = mo_coeff.shape[2]

# 按自旋通道分别取占据轨道。
# 由于 alpha/beta 占据数可能不同 (nocc_a != nocc_b)，无法合并为单个 ndarray，使用 list 存储。
mocca = mo_coeff[α][:, mo_occ[α] > 0]
moccb = mo_coeff[β][:, mo_occ[β] > 0]
mocc = [mocca, moccb]
nocc = [mocca.shape[1], moccb.shape[1]]

# 在 RHF 中 mocc_2 = mocc * sqrt(occ=2)；UHF 中 occ=1 因此 sqrt(occ)=1，
# 也即 mocc_2 直接等于按自旋分量的占据轨道系数，保留同名以与 RHF 公式对齐。
mocc_2 = [mocca, moccb]

# 各自旋的轨道能量（同样按 list 存储）
eocc = [mo_energy[α][mo_occ[α] > 0], mo_energy[β][mo_occ[β] > 0]]
evir = [mo_energy[α][mo_occ[α] == 0], mo_energy[β][mo_occ[β] == 0]]

# UHF 密度矩阵（shape [2, nao, nao]），以及总密度 dm0_s（K 不会用到，但保留以便参考）
dm0 = np.zeros((2, nao, nao))
dm0[α] = mocca @ mocca.T
dm0[β] = moccb @ moccb.T
dm0_s = dm0.sum(axis=0)

natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao

# 详细分解

实现原则与 RHF 完全一致：每个子项先按 einsum 计算一个与原子无关的 `dbas_*` 张量，再通过双重 `(A, B)` 循环切片求和得到 `[natm, natm, 3, 3]` 的原子贡献，最后用 `np.allclose` 核验。

**UHF 的具体处理**：交换项按自旋分别计算，公式形式上对 `mocc[α]` 和 `mocc[β]` 各跑一遍，最后两个自旋贡献相加。把按自旋的 einsum 写到一个辅助函数 / 循环里可以避免代码重复。

In [7]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])
int3c2e_ipip1 = _int3c_wrapper(mol, aux, "int3c2e_ipip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipvip1 = _int3c_wrapper(mol, aux, "int3c2e_ipvip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ip1ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip1ip2", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipip2 = _int3c_wrapper(mol, aux, "int3c2e_ipip2", "s1")().reshape([3, 3, nao, nao, naux])
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int2c2e_ipip1 = aux.intor("int2c2e_ipip1").reshape([3, 3, naux, naux])
int2c2e_ip1ip2 = aux.intor("int2c2e_ip1ip2").reshape([3, 3, naux, naux])

### K (basis_2nd)

In [8]:
# (10|0)(0|10), part a — 按 α/β 两个自旋分别计算后求和
de_K20_1a = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(aoslices):
        for B, (_, _, p0B, p1B) in enumerate(aoslices):
            de_K20_1a[A, B] += 2 * np.einsum("tsuk -> ts", dbas[:, :, p0A:p1A, p0B:p1B])

In [9]:
# (10|0)(0|10), part b
de_K20_1b = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tuvP, PQ, sklQ, ui, vj, kj, li -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(aoslices):
        for B, (_, _, p0B, p1B) in enumerate(aoslices):
            de_K20_1b[A, B] += 2 * np.einsum("tsuk -> ts", dbas[:, :, p0A:p1A, p0B:p1B])

In [10]:
# (11|0)(0|00)
de_K20_2 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsuv", int3c2e_ipvip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(aoslices):
        for B, (_, _, p0B, p1B) in enumerate(aoslices):
            de_K20_2[A, B] += 2 * np.einsum("tsuv -> ts", dbas[:, :, p0A:p1A, p0B:p1B])

In [11]:
# (20|0)(0|00)
de_K20_3 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsuv", int3c2e_ipip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(aoslices):
        de_K20_3[A, A] += 2 * np.einsum("tsuv -> ts", dbas[:, :, p0A:p1A])

In [12]:
de_K20_recap = de_K20_1a + de_K20_1b + de_K20_2 + de_K20_3
assert np.allclose(de_K20_recap, de_K20, atol=1e-5, rtol=1e-4)

### K (basis_1st_aux_1st)

In [13]:
# (10|1)(0|0)(0|00)
de_K11_1 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tsuvP, PQ, klQ, vi, li, uj, kj -> tsuP", int3c2e_ip1ip2, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(aoslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K11_1[A, B] += 2 * np.einsum("tsuP -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K11_1 += de_K11_1.transpose(1, 0, 3, 2)

In [14]:
# (10|0)(0|1)(0|00)
de_K11_2 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsuR", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(aoslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K11_2[A, B] += 2 * np.einsum("tsuR -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K11_2 += de_K11_2.transpose(1, 0, 3, 2)

In [15]:
# (10|0)(1|0)(0|00)
de_K11_3 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsuQ", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(aoslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K11_3[A, B] += -2 * np.einsum("tsuQ -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K11_3 += de_K11_3.transpose(1, 0, 3, 2)

In [16]:
# (10|0)(0|0)(1|00)
de_K11_4 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsuQ", int3c2e_ip1, int2c2e_inv, int3c2e_ip2, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(aoslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K11_4[A, B] += 2 * np.einsum("tsuQ -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K11_4 += de_K11_4.transpose(1, 0, 3, 2)

In [17]:
de_K11_recap = de_K11_1 + de_K11_2 + de_K11_3 + de_K11_4
assert np.allclose(de_K11_recap, de_K11, atol=1e-5, rtol=1e-4)

### K (aux_2nd)

In [18]:
# (00|2)(0|00)
de_K02_1 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsP", int3c2e_ipip2, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        de_K02_1[A, A] += np.einsum("tsP -> ts", dbas[:, :, p0A:p1A])

In [19]:
# (00|0)(2|0)(0|00)
de_K02_2 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("uvP, PQ, tsQR, RS, klS, ui, vj, ki, lj -> tsQ", int3c2e, int2c2e_inv, int2c2e_ipip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        de_K02_2[A, A] += -1 * np.einsum("tsQ -> ts", dbas[:, :, p0A:p1A])

In [20]:
# (00|0)(1|1)(0|00)
de_K02_3a = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("uvP, PQ, tsQR, RS, klS, ui, vj, ki, lj -> tsQR", int3c2e, int2c2e_inv, int2c2e_ip1ip2, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K02_3a[A, B] += -0.5 * np.einsum("tsQR -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K02_3a += de_K02_3a.transpose(1, 0, 3, 2)

In [21]:
# (00|0)(1|0)(0|1)(0|00)
de_K02_3b = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, ui, vj, ki, lj -> tsQT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K02_3b[A, B] += -0.5 * np.einsum("tsQT -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K02_3b += de_K02_3b.transpose(1, 0, 3, 2)

In [22]:
# (00|1)(1|0)(0|00)
de_K02_4 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsPQ", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K02_4[A, B] += -1 * np.einsum("tsPQ -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K02_4 += de_K02_4.transpose(1, 0, 3, 2)

In [23]:
# (00|1)(1|00)
de_K02_5 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsPQ", int3c2e_ip2, int2c2e_inv, int3c2e_ip2, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K02_5[A, B] += 0.5 * np.einsum("tsPQ -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K02_5 += de_K02_5.transpose(1, 0, 3, 2)

In [24]:
# (00|0)(0|1)(1|0)(0|00)
de_K02_6 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("uvP, PQ, tRQ, RS, sST, TU, klU, ui, vj, ki, lj -> tsRS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K02_6[A, B] += 0.5 * np.einsum("tsRS -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K02_6 += de_K02_6.transpose(1, 0, 3, 2)

In [25]:
# (00|1)(0|1)(0|00)
de_K02_7 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("tuvP, PQ, sRQ, RS, klS, ui, vj, ki, lj -> tsPR", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K02_7[A, B] += -1 * np.einsum("tsPR -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K02_7 += de_K02_7.transpose(1, 0, 3, 2)

In [26]:
# (00|0)(1|0)(1|0)(0|00)
de_K02_8 = np.zeros((natm, natm, 3, 3))
for x in (α, β):
    dbas = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, ui, vj, ki, lj -> tsQS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2[x], mocc_2[x], mocc_2[x], mocc_2[x])
    for A, (_, _, p0A, p1A) in enumerate(auxslices):
        for B, (_, _, p0B, p1B) in enumerate(auxslices):
            de_K02_8[A, B] += 1 * np.einsum("tsQS -> ts", dbas[:, :, p0A:p1A, p0B:p1B])
de_K02_8 += de_K02_8.transpose(1, 0, 3, 2)

In [27]:
de_K02_recap = de_K02_1 + de_K02_2 + de_K02_3a + de_K02_3b + de_K02_4 + de_K02_5 + de_K02_6 + de_K02_7 + de_K02_8
assert np.allclose(de_K02_recap, de_K02, atol=1e-5, rtol=1e-4)

## 存储到文件

In [28]:
dat = dict(np.load("nh3_u_hf_decomp.npz"))
dat.update({
    # de_K20
    "de_K20_1a": de_K20_1a,
    "de_K20_1b": de_K20_1b,
    "de_K20_2": de_K20_2,
    "de_K20_3": de_K20_3,
    # de_K11
    "de_K11_1": de_K11_1,
    "de_K11_2": de_K11_2,
    "de_K11_3": de_K11_3,
    "de_K11_4": de_K11_4,
    # de_K02
    "de_K02_1": de_K02_1,
    "de_K02_2": de_K02_2,
    "de_K02_3a": de_K02_3a,
    "de_K02_3b": de_K02_3b,
    "de_K02_4": de_K02_4,
    "de_K02_5": de_K02_5,
    "de_K02_6": de_K02_6,
    "de_K02_7": de_K02_7,
    "de_K02_8": de_K02_8,
})
np.savez("nh3_u_hf_decomp.npz", **dat)